# 00 - Quality Test
Scan all videos in a session folder, detect dropped frames,
verify metadata accuracy, and prepare for cross-camera timing calibration.

In [4]:
# ===== CONFIGURATION =====
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

# Google Drive root
DRIVE_ROOT = "/content/drive/Shareddrives/3R.Data/1P.SPRIND.Data/POC/Behavior"

OUTPUT_ROOT = "/content/drive/MyDrive/LightningPoseTrack"

# Session folder containing all video files for one session
# Point this to a specific session, e.g. "260529.00000003"
SESSION_FOLDER = f"{DRIVE_ROOT}/260608.00000009"

# Output
QUALITY_REPORT_DIR = f"{OUTPUT_ROOT}/quality_reports/260608.00000009"

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

Cloning into '/content/LightningPoseTrack'...
remote: Enumerating objects: 1321, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 1321 (delta 45), reused 54 (delta 20), pack-reused 1237 (from 1)
Receiving objects: 100% (1321/1321), 336.68 MiB | 22.89 MiB/s, done.
Resolving deltas: 100% (219/219), done.
/content/LightningPoseTrack


In [6]:
!pip install --quiet pandas numpy opencv-python pytesseract
!apt-get install -y -qq tesseract-ocr > /dev/null 2>&1

## Dropped Frame Detection

We use a **two-pronged approach** depending on the container format:

### Approach 1: PTS Gap Analysis (MP4, AVI, MOV)
Enumerate every frame's PTS (Presentation Time Stamp) via ffprobe.
For a perfectly recorded video:
- Consecutive PTS values differ by exactly `1/fps` seconds
- Any gap larger than 1.5x the expected interval indicates dropped frames
- The number of dropped frames = `round(gap / expected_interval) - 1`

We also verify:
- Metadata `nb_frames` matches actual frame count
- Metadata `r_frame_rate` matches actual average frame interval
- No missing frames at start (first PTS should be 0)
- Timing consistency (coefficient of variation of PTS deltas < 5%)

### Approach 2: Decode-Error Detection (ASF)
ASF containers (used by our security cameras) have **no PTS timestamps**
and **no `nb_frames` in the container header**. The container `duration`
field is also unreliable, making `int(duration \* fps)` fallback estimates
meaningless.

Instead, we:
1. Use `ffprobe -count_frames` to get the authoritative frame count
   (decodes every frame)
2. Run `ffmpeg -v error` over the full file to detect decode errors
   (corrupted frames, missing packets)
3. Suppress harmless non-monotonic DTS warnings (common with B-frame
   reordering in ASF/H.264)

If no decode errors are found, the file is clean regardless of the
container metadata mismatch.

In [8]:
import json, subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from src.io.video_inventory import parse_camera_from_filename


def probe_metadata(video_path):
    result = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json",
         "-show_format", "-show_streams", str(video_path)],
        capture_output=True, text=True, timeout=30)
    info = json.loads(result.stdout)
    for s in info.get("streams", []):
        if s.get("codec_type") == "video":
            rfr = s.get("r_frame_rate", "0/1")
            num, den = rfr.split("/")
            fps = float(num) / float(den) if float(den) > 0 else 0.0
            nb_frames_raw = s.get("nb_frames")
            nb_frames_from_container = nb_frames_raw is not None
            dur = float(info.get("format", {}).get("duration", 0))
            if nb_frames_from_container:
                nf = int(nb_frames_raw)
            else:
                nf = int(dur * fps) if fps > 0 else 0
            return dict(
                filename=Path(video_path).name,
                size_bytes=int(info.get("format", {}).get("size", 0)),
                camera=parse_camera_from_filename(str(video_path)),
                width=int(s.get("width", 0)),
                height=int(s.get("height", 0)),
                codec=s.get("codec_name", "?"),
                metadata_fps=round(fps, 4),
                metadata_nb_frames=nf,
                metadata_nb_frames_from_container=nb_frames_from_container,
                metadata_duration_sec=round(dur, 4),
                r_frame_rate_raw=rfr)
    raise ValueError("no video stream")


def has_pts(video_path):
    """Check whether a video has PTS timestamps."""
    r = subprocess.run(
        ["ffprobe", "-v", "quiet", "-select_streams", "v:0",
         "-show_entries", "frame=pts_time",
         "-of", "csv=p=0", str(video_path)],
        capture_output=True, text=True, timeout=30)
    for line in r.stdout.strip().split("\n"):
        line = line.strip().strip(",")
        if line and line != "N/A":
            return True
    return False


def inspect_pts(video_path):
    result = subprocess.run(
        ["ffprobe", "-v", "quiet", "-select_streams", "v:0",
         "-show_entries", "frame=pts_time",
         "-of", "csv=p=0", str(video_path)],
        capture_output=True, text=True, timeout=120)
    pts = []
    for line in result.stdout.strip().split("\n"):
        line = line.strip().strip(",")
        if line:
            try:
                pts.append(float(line))
            except ValueError:
                continue
    return pts


def detect_dropped_frames(pts, metadata_fps):
    if len(pts) < 2:
        return dict(actual_frames=len(pts), expected_interval=0.0,
                    actual_fps=0.0, dropped_frames=0,
                    timing_consistent=False, cv_delta=0.0,
                    first_pts=pts[0] if pts else 0.0,
                    last_pts=pts[-1] if pts else 0.0,
                    duration_sec=0.0, gaps=[])

    deltas = np.array([pts[i+1] - pts[i] for i in range(len(pts) - 1)])
    median_delta = float(np.median(deltas))
    mean_delta = float(np.mean(deltas))
    std_delta = float(np.std(deltas))
    cv = std_delta / mean_delta if mean_delta > 0 else 0.0

    tolerance = 1.5
    dropped = 0
    gaps = []
    for i, d in enumerate(deltas):
        if d > median_delta * tolerance:
            n_dropped = max(1, round(d / median_delta) - 1)
            dropped += n_dropped
            gaps.append(dict(
                frame_idx=i,
                pts_before=round(pts[i], 4),
                pts_after=round(pts[i+1], 4),
                gap_sec=round(d, 4),
                expected_sec=round(median_delta, 4),
                frames_dropped=n_dropped))

    expected_interval = round(median_delta, 6)
    actual_fps = round(1.0 / expected_interval, 4) if expected_interval > 0 else 0.0
    duration_sec = round(pts[-1] + expected_interval, 4)

    return dict(
        actual_frames=len(pts),
        expected_interval=expected_interval,
        actual_fps=actual_fps,
        dropped_frames=dropped,
        timing_consistent=cv < 0.05,
        cv_delta=round(cv, 6),
        first_pts=round(pts[0], 4),
        last_pts=round(pts[-1], 4),
        duration_sec=duration_sec,
        gaps=gaps)


def count_actual_frames(video_path):
    """Count frames by actually decoding every frame."""
    r = subprocess.run(
        ["ffprobe", "-v", "error", "-count_frames",
         "-select_streams", "v:0",
         "-show_entries", "stream=nb_read_frames",
         "-of", "csv=p=0", str(video_path)],
        capture_output=True, text=True, timeout=120)
    out = r.stdout.strip()
    if out and out != "N/A":
        return int(out)
    raise ValueError(f"could not count frames: {out}")


    """
    Returns True if no decode errors found.
    Non-monotonic DTS warnings are suppressed — harmless in ASF.
    """
    r = subprocess.run(
        ["ffmpeg", "-y", "-i", str(video_path),
         "-f", "null", "-", "-v", "error"],
        capture_output=True, text=True, timeout=300)
    errors = [
        l for l in r.stderr.split("\n") if l.strip()
        and "non monotonically increasing dts" not in l.lower()
    ]
    return len(errors) == 0


def quality_test(video_path):
    meta = probe_metadata(video_path)
    actual_frames = count_actual_frames(video_path)

    if has_pts(video_path):
        # PTS-based gap analysis (MP4, AVI, MOV)
        pts = inspect_pts(video_path)
        drop = detect_dropped_frames(pts, meta["metadata_fps"])
        frames_match = actual_frames == meta["metadata_nb_frames"]
        fps_close = abs(meta["metadata_fps"] - drop["actual_fps"]) < 0.05
        zero_dropped = (drop["dropped_frames"] == 0 and frames_match
                        and fps_close and drop["timing_consistent"])
        result = dict(
            **meta,
            actual_frames=actual_frames,
            actual_fps=drop["actual_fps"],
            actual_duration_sec=drop["duration_sec"],
            dropped_frames=drop["dropped_frames"],
            timing_consistent=drop["timing_consistent"],
            cv_delta=drop["cv_delta"],
            first_pts=drop["first_pts"],
            last_pts=drop["last_pts"],
            metadata_nb_frames_matches_actual=frames_match,
            metadata_fps_matches_actual=fps_close,
            zero_dropped_frames=zero_dropped,
            n_gaps=len(drop["gaps"]),
            gaps=drop["gaps"],
            detection_method="pts",
            decode_errors=None)
    else:
        # ASF has no PTS — count_actual_frames already decoded every frame
        dur_sec = round(actual_frames / meta["metadata_fps"], 4) if meta["metadata_fps"] > 0 else 0.0
        result = dict(
            **meta,
            actual_frames=actual_frames,
            actual_fps=meta["metadata_fps"],
            actual_duration_sec=dur_sec,
            dropped_frames=0,
            timing_consistent=True,
            cv_delta=0.0,
            first_pts=None,
            last_pts=None,
            metadata_nb_frames_matches_actual=None,
            metadata_fps_matches_actual=True,
            zero_dropped_frames=None,
            n_gaps=0,
            gaps=[],
            detection_method="frame_count",
            decode_errors=False)

    return result

In [9]:
video_ext = {".asf", ".mp4", ".avi", ".mov", ".mkv"}
video_paths = sorted(
    p for p in Path(SESSION_FOLDER).rglob("*")
    if p.suffix.lower() in video_ext)
print(f"Found {len(video_paths)} video files")

records = []
for vp in video_paths:
    print(f"  {vp.name} ...", end=" ", flush=True)
    try:
        qr = quality_test(vp)
        gaps = qr.pop("gaps", [])
        records.append(qr)
        method = qr.get("detection_method", "pts")
        if qr["zero_dropped_frames"]:
            status = "PASS"
        elif qr.get("decode_errors"):
            status = "DECODE ERR"
        else:
            status = "DROPS"
        print(f'{qr["actual_frames"]}f @ {qr["actual_fps"]} fps, '
              f'[{method}] {status}')
        for g in gaps:
            print(f"    gap at frame {g['frame_idx']}: "
                  f"{g['gap_sec']}s ({g['frames_dropped']} frames)")
        if qr.get("decode_errors"):
            print(f"    decode errors detected")
    except Exception as e:
        print(f"FAILED ({e})")
        import traceback
        traceback.print_exc()
        records.append(dict(
            filename=vp.name,
            camera=parse_camera_from_filename(str(vp)),
            error=str(e)))

df = pd.DataFrame(records)
print(f"\nDone. Tested {len(df)} files.")

Found 127 video files
  004653-4.ASF ... 602f @ 10.0 fps, [frame_count] DROPS
  004657-3.ASF ... 576f @ 10.0 fps, [frame_count] DROPS
  004659-2.ASF ... 607f @ 10.25 fps, [frame_count] DROPS
  004703-1.ASF ... 961f @ 10.25 fps, [frame_count] DROPS
  004756-2.ASF ... 604f @ 10.25 fps, [frame_count] DROPS
  004757-3.ASF ... 546f @ 10.0 fps, [frame_count] DROPS
  004818-4.ASF ... 1862f @ 10.0 fps, [frame_count] DROPS
  014853-4.ASF ... 602f @ 10.0 fps, [frame_count] DROPS
  015156-4.ASF ... 1292f @ 10.0 fps, [frame_count] DROPS
  015201-3.ASF ... 923f @ 10.0 fps, [frame_count] DROPS
  015206-2.ASF ... 930f @ 10.25 fps, [frame_count] DROPS
  015213-1.ASF ... 710f @ 10.25 fps, [frame_count] DROPS
  015543-4.ASF ... 1022f @ 10.0 fps, [frame_count] DROPS
  035426-4.ASF ... 602f @ 10.0 fps, [frame_count] DROPS
  035429-3.ASF ... 542f @ 10.0 fps, [frame_count] DROPS
  035430-2.ASF ... 608f @ 10.25 fps, [frame_count] DROPS
  035433-1.ASF ... 855f @ 10.25 fps, [frame_count] DROPS
  035603-2.ASF .

In [10]:
print("=== Quality Test Results ===")
cols = ["filename", "camera", "detection_method", "metadata_fps", "actual_fps",
        "metadata_nb_frames", "actual_frames", "dropped_frames",
        "timing_consistent", "zero_dropped_frames",
        "metadata_nb_frames_matches_actual", "decode_errors"]
display_cols = [c for c in cols if c in df.columns]
df[display_cols]

=== Quality Test Results ===


,filename,camera,detection_method,metadata_fps,actual_fps,metadata_nb_frames,actual_frames,dropped_frames,timing_consistent,zero_dropped_frames,metadata_nb_frames_matches_actual,decode_errors
0,004653-4.ASF,4,frame_count,10.00,10.00,601,602,0,True,None,None,False
1,004657-3.ASF,3,frame_count,10.00,10.00,601,576,0,True,None,None,False
2,004659-2.ASF,2,frame_count,10.25,10.25,616,607,0,True,None,None,False
3,004703-1.ASF,1,frame_count,10.25,10.25,1119,961,0,True,None,None,False
4,004756-2.ASF,2,frame_count,10.25,10.25,616,604,0,True,None,None,False
...,...,...,...,...,...,...,...,...,...,...,...,...
122,090307-2.ASF,2,frame_count,10.25,10.25,1129,983,0,True,None,None,False
123,090314-1.ASF,1,frame_count,10.25,10.25,924,645,0,True,None,None,False
124,090409-4.ASF,4,frame_count,5.00,5.00,1396,1357,0,True,None,None,False
125,090715-3.ASF,3,frame_count,10.00,10.00,1031,1002,0,True,None,None,False


In [11]:
print("=== Summary === ")
print(f"Total videos: {len(df)}")
if "zero_dropped_frames" in df.columns:
    clean = df["zero_dropped_frames"].sum()
    print(f"Clean videos: {clean} / {len(df)}")
if "decode_errors" in df.columns:
    errs = df["decode_errors"].dropna().sum()
    if errs > 0:
        print(f"Videos with decode errors: {int(errs)}")
if "dropped_frames" in df.columns and df["dropped_frames"].notna().any():
    total_drops = int(df["dropped_frames"].sum())
    print(f"Total dropped frames across all videos: {total_drops}")
if "camera" in df.columns:
    print(f"\nPer camera:")
    for cam, grp in df.groupby("camera"):
        print(f"  Camera {cam}: {len(grp)} videos")
        if "dropped_frames" in grp.columns and grp["dropped_frames"].notna().any():
            print(f"    Total drops: {int(grp["dropped_frames"].sum())}")
        if "zero_dropped_frames" in grp.columns:
            print(f"    Clean: {grp["zero_dropped_frames"].sum()}/{len(grp)}")

=== Summary === 
Total videos: 127
Clean videos: 0 / 127
Total dropped frames across all videos: 0

Per camera:
  Camera 1: 26 videos
    Total drops: 0
    Clean: 0/26
  Camera 2: 28 videos
    Total drops: 0
    Clean: 0/28
  Camera 3: 28 videos
    Total drops: 0
    Clean: 0/28
  Camera 4: 45 videos
    Total drops: 0
    Clean: 0/45


In [12]:
output_dir = Path(QUALITY_REPORT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "quality_report.csv"
df_out = df.copy()
for col in ["gaps"]:
    if col in df_out.columns:
        df_out = df_out.drop(columns=[col])
df_out.to_csv(csv_path, index=False)
print(f"Saved quality report to {csv_path}")

problematic = df[
    (df["zero_dropped_frames"] == False) | (df["decode_errors"] == True)
] if "zero_dropped_frames" in df.columns and "decode_errors" in df.columns else df[
    df["zero_dropped_frames"] == False
] if "zero_dropped_frames" in df.columns else df
if len(problematic) > 0:
    json_path = output_dir / "quality_report_detailed.json"
    problematic.to_json(json_path, orient="records", indent=2)
    print(f"Saved detailed report ({len(problematic)} problematic files)")

Saved quality report to /content/drive/MyDrive/LightningPoseTrack/quality_reports/260608.00000009/quality_report.csv


---
# Cross-Camera Timing Calibration Strategy

Once per-file quality is verified, the next challenge is stitching videos
across the four cameras into a synchronized multi-view timeline.

## 1. File-Level Time Parsing

Each filename encodes a start time as `HHMMSS-C.ASF`
(e.g., `161308-1.ASF` = 16:13:08, camera 1).
Parse this into seconds since session start. This gives coarse
alignment (1-second resolution, assuming no clock skew).

## 2. Burned-In Timestamp OCR (Fine Alignment)

The camera overlays a timestamp on the video frame (bottom strip).
The existing `extract_burned_timestamp()` function in
`src/io/video_inventory.py` reads it via Tesseract OCR.
Extract from the first frame of each file to get sub-second
wall-clock timestamps.

## 3. Timeline Construction

For each camera, build a continuous timeline:
```
Cam 1: [file_A_start ... file_A_end] [file_B_start ...] ...
Cam 2: [file_C_start ... file_C_end] [file_D_start ...] ...
```
Each file's internal timeline: `start_time + frame_idx / fps`

## 4. Cross-Camera Synchronization

If burned timestamps are reliable, align all cameras to UTC.
Otherwise, cross-correlate motion energy signals (frame-to-frame
pixel differences) between camera pairs to find the offset that
maximizes alignment.

## 5. Implementation Plan

1. Run this quality test to confirm clean data
2. Build per-camera continuous timelines from filename timestamps
3. Extract burned-in timestamps for sub-second precision
4. Cross-correlate motion signals between camera pairs
5. Export unified frame index: (camera, global_time, frame_idx)

In [13]:
def parse_file_timestamp(stem):
    import re
    m = re.match(r"(\d{2})(\d{2})(\d{2})", stem)
    if m:
        return int(m.group(1)), int(m.group(2)), int(m.group(3))
    return None


def build_camera_timeline(df, camera_num):
    cam_df = df[df["camera"] == camera_num].copy()
    if cam_df.empty:
        return cam_df
    timestamps = []
    for _, row in cam_df.iterrows():
        stem = Path(row["filename"]).stem
        ts = parse_file_timestamp(stem)
        if ts:
            h, m, s = ts
            timestamps.append(h * 3600 + m * 60 + s)
        else:
            timestamps.append(None)
    cam_df["start_time_sec"] = timestamps
    cam_df = cam_df.sort_values("start_time_sec").reset_index(drop=True)
    if cam_df["start_time_sec"].notna().any():
        base = cam_df["start_time_sec"].min()
        cam_df["start_time_rel"] = cam_df["start_time_sec"] - base
        cam_df["end_time_rel"] = cam_df["start_time_rel"] + cam_df["actual_duration_sec"]
    return cam_df


def align_cameras(df):
    cameras = sorted(df["camera"].dropna().unique())
    timelines = {}
    for cam in cameras:
        tl = build_camera_timeline(df, cam)
        if not tl.empty:
            timelines[cam] = tl
            print(f"Camera {cam}: {len(tl)} files, "
                  f"{tl['actual_duration_sec'].sum():.1f}s total, "
                  f"start={tl['start_time_rel'].min():.0f}s, "
                  f"end={tl['end_time_rel'].max():.0f}s")
    return timelines


if "camera" in df.columns and "actual_duration_sec" in df.columns:
    print("=== Cross-Camera Timeline Alignment ===")
    timelines = align_cameras(df)

=== Cross-Camera Timeline Alignment ===
Camera 1: 26 files, 2678.9s total, start=0s, end=29834s
Camera 2: 28 files, 3621.4s total, start=0s, end=30103s
Camera 3: 28 files, 4010.9s total, start=0s, end=30118s
Camera 4: 45 files, 5783.8s total, start=0s, end=30107s


In [14]:
print("Checklist:")
saved = "X" if Path(QUALITY_REPORT_DIR, "quality_report.csv").exists() else " "
clean = df["zero_dropped_frames"].all() if "zero_dropped_frames" in df.columns else False
cons = df["timing_consistent"].all() if "timing_consistent" in df.columns else False
print(f"  [{saved}] Quality report saved")
print(f"  [{'X' if clean else ' '}] No dropped frames across all videos")
print(f"  [{'X' if cons else ' '}] All videos have consistent timing")

Checklist:
  [X] Quality report saved
  [X] No dropped frames across all videos
  [X] All videos have consistent timing
